# Imports

In [1]:


import pycountry

import os, sys, json, argparse, logging, requests
sys.path.append("/Users/danielrubibreton/Documents/important/toolkit/")
from app_tkns import TMDB_key, TMDB_read_access_token
from typing import List, Dict, Any, Optional, TypedDict, NotRequired, Annotated
from enum import Enum
import pandas as pd
from langgraph.graph import StateGraph, START, END, MessagesState, add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, AnyMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field, conlist, constr, model_validator


from IPython.display import Markdown, display

from textwrap import dedent as ded
dedent = lambda x: ded(x.strip())



/Users/danielrubibreton/Desktop/PythonStuff/agents/germandutch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Functions

In [3]:
def rename_country(country):
    name = pycountry.countries.get(alpha_2=country).name
    flag = pycountry.countries.get(alpha_2=country).flag

    return name.title()
    return f"{name} {flag}"

def filter_subscribed(x):
    for i in SUBSCRIBED:
        if i in x:
            return True
    return False

from string import ascii_uppercase

In [4]:
ALL_FLAGS = pd.Series({
    f"{a}{b}": pycountry.countries.get(alpha_2=f"{a}{b}")
    for a in ascii_uppercase
    for b in ascii_uppercase
}).dropna().apply(lambda x: f"{x.name} ({x.flag})".upper() )

ALL_FLAGS.shape

(249,)

In [5]:
len(list(pycountry.countries))

249

In [6]:
ALL_FLAGS.loc[
    ALL_FLAGS.str.contains("NOR")
]

MK                         NORTH MACEDONIA (🇲🇰)
MP                NORTHERN MARIANA ISLANDS (🇲🇵)
NF                          NORFOLK ISLAND (🇳🇫)
NO                                  NORWAY (🇳🇴)
UM    UNITED STATES MINOR OUTLYING ISLANDS (🇺🇲)
dtype: object

In [7]:
def tmdb_table(results) -> pd.DataFrame:
    """Get a table of the info of providers per country"""
    if results:
    
        sub_df = (
            pd.DataFrame(
                {
                    rename_country(country):{k:[i["provider_name"] for i in v] 
                        for k,v in results[country].items() 
                                             if k in ["ads","flatrate","free"]#["ads","rent","flatrate","buy","free"] 
                                        } 
                        for country in results.keys()
                }
            )
        )
    
        if sub_df.empty:
            sub_df = (
                pd.DataFrame(
                    {
                        rename_country(country):{k:[i["provider_name"] for i in v] 
                            for k,v in results[country].items() 
                                                 if k in ["ads","rent","flatrate","buy","free"] 
                                            } 
                            for country in results.keys()
                    }
                )
            )
            
    
    
        return json.dumps(
            sub_df
            .stack().explode().sort_index()
                .reset_index()
                .set_axis(["Type","Country","Provider"],axis=1)
            .assign(
                Provider = lambda df: df["Provider"].apply(lambda x: str(x).upper().strip() )
            )
            .loc[lambda df: df["Provider"].apply(filter_subscribed)]
            .pivot_table(
                index = ["Type","Provider"],
                values = "Country",
                aggfunc=list
            )
            .groupby(level=0)  # group by "Type"
            .apply(lambda g: g.droplevel(0).to_dict()["Country"])  # nested dict by Provider
            .to_dict(),
            indent=4
        )
    return pd.DataFrame()

In [8]:
#@tool
def tmdb_search_movie(title: str, year: Optional[int] = None, director: Optional[str] = None) -> Dict[str, Any]:
    """Search TMDB by title (optional year/director). Returns up to 10 candidates."""
    if not TMDB_API_KEY:
        return {"error": "TMDB_API_KEY not set"}
    params = {"api_key": TMDB_API_KEY, "query": title, "include_adult": "false"}
    if year: params["year"] = year
    data = _rq(f"{TMDB_BASE}/search/movie", params)
    cands = []
    for r in (data.get("results") or [])[:10]:
        cands.append({
            "id": r.get("id"),
            "title": r.get("title"),
            "original_language": (r.get("original_language") or ""),
            "release_year": (r.get("release_date") or "")[:4],
            "overview": r.get("overview"),
        })
    #return {"candidates": cands, "instruction": "Pick the single best match."}
    return cands


#@tool
def tmdb_watch_providers(movie_id: int) -> Dict[str, Any]:
    """Get TMDB Watch Providers (per country) for a movie_id."""
    if not TMDB_API_KEY:
        return {"error": "TMDB_API_KEY not set"}
    data = _rq(f"{TMDB_BASE}/movie/{movie_id}/watch/providers", {"api_key": TMDB_API_KEY})

    results = data.get("results", {})
    if results:
        return tmdb_table(results)
    
    return ""

## Lang Graph Functions

In [9]:
def parse_movie_node(state):
    class TMDBSearchMovieInput(BaseModel):
        """Input schema for tmdb_search_movie."""
        title: str = Field(..., description="Movie title to search for")
        year: Optional[int] = Field(None, description="null if not mentioned by the user")
        director: Optional[str] = Field(None, description="null if not mentioned by the user")

        # Optional sanity check (LLM guardrail)
        @model_validator(mode="after")
        def enforce_nulls(cls, values):
            # If year or director are empty strings, coerce to None
            if not values.year:
                values.year = None
            if not values.director:
                values.director = None
            return values

    system_prompt = dedent("""
    You must extract 'title', 'year', and 'director' from the user query.
    - If year is not explicitly mentioned, return year = null.
    - If director is not explicitly mentioned, return director = null.
    - No extra text, no explanations.
    Example outputs:
      {"title": "Inception", "year": 2010, "director": "Christopher Nolan"}
      {"title": "Titanic", "year": null, "director": null}
    """)

    try:
        response = llm.with_structured_output(TMDBSearchMovieInput).invoke(
            [
                SystemMessage(content=system_prompt),
                state["messages"][-1]
            ]
        )
    except Exception as e:
        return {"next_action": "return", "error": str(e)}

    return {'user_movie': response.model_dump()}

In [10]:
def choose_movie_node(state):
    user_movie = state.get("user_movie", {})

    if not user_movie:
        return {"error": "No retrieved movie"}
        
    movie_candidates = tmdb_search_movie(
        title    = user_movie.get("title",""), 
        year     = user_movie.get("year",None),
        director = user_movie.get("director",None), 
    )

    class TMDBPickMovie(BaseModel):
        """Pick exactly one candidate by id, or null if no suitable match."""
        movie_id: int = Field(
            ...,
            description="One of the candidate IDs shown below. Use null if none match the user intent."
        )

    system_prompt = dedent("""
        You are selecting exactly ONE movie from the provided candidates.
        Rules:
        - Return a JSON object that matches the schema.
        - 'movie_id' MUST be one of the candidate IDs shown. If no candidate fits, set id = null.
        - No prose beyond fields in the schema.
    """)

    user_query = state.get("messages", [HumanMessage(content="")])[-1].content if "messages" in state else ""

    human_content = dedent(f"""
    User query: {user_query}

    Candidates (JSON list):
    {movie_candidates}
    """)

    # 3) ask the LLM for a structured choice
    try:
        pick: TMDBPickMovie = llm.with_structured_output(TMDBPickMovie).invoke(
            [SystemMessage(content=system_prompt), HumanMessage(content=human_content)]
        )
    except Exception as e:
        return {"movie_id": None, "error": f"LLM error: {e}"}

    # 4) hard validate: id must be in candidates
    valid_ids = {c["id"] for c in movie_candidates if c.get("id") is not None}
    chosen_id = pick.movie_id if (pick.movie_id in valid_ids) else None

    if chosen_id is not None:
        retrived_movie = [i for i in movie_candidates if i["id"]==chosen_id][0]
        providers_for_movie = tmdb_watch_providers(movie_id=retrived_movie["id"])

        if providers_for_movie is None:
            return {
                "retrived_movie": retrived_movie,
                "messages": [
                    ToolMessage(content=str(user_movie),     tool_call_id="Parse Tool"),
                    ToolMessage(content=str(retrived_movie), tool_call_id="Movie db Tool"),
                    ToolMessage(content=str(retrived_movie), tool_call_id="Retriever")
                ]
            }
        else:
            return {
                "retrived_movie": retrived_movie,
                "providers": providers_for_movie,
                "messages": [
                    ToolMessage(content=str(user_movie),     tool_call_id="Parse Tool"),
                    ToolMessage(content=str(retrived_movie), tool_call_id="Movie db Tool"),
                    ToolMessage(content=providers_for_movie, tool_call_id="Provider Tool")
                ]
            }
        
    else:
        return {"error": "Movie not found"}

In [11]:
def response_wrap_node(state):
    messages = state.get("messages",[])

    system_prompt = dedent("""
    You produce the final user-facing answer for an AI system that searches in which services and which countries to stream the movie asked.
    ## Context
    - The given data is already filtering the subscription the user has
    - The user location is the Netherlands, so that is the only Non-VPN option that exists
    
    ## Constraints:
    - It doesn't matter what is the user's country. They will use a VPN
    - Mention ALL countries where the movie is available
    - Mention all available options to see the movie (only if info is available)
    - If nothing is available anywhere, say so.
    - Do not invent providers or countries not in the input.
    
    """)

    response = llm.invoke(messages+[SystemMessage(content=system_prompt)])
    return {
        "messages": [response]
    }

## Agent Build

In [12]:
# ---- State schema used by the three nodes ----
class MovieState(TypedDict, total=False):
    # conversation
    messages: Annotated[List[BaseMessage], add_messages]

    # parsed from user by parse_movie_node
    user_movie: Dict[str, Optional[Any]]  # {"title": str, "year": Optional[int], "director": Optional[str]}

    # chosen by choose_movie_node
    retrived_movie: Dict[str, Any]        # {"id": int, "title": str, "original_language": str, "release_year": str, "overview": str}
    providers: Dict[str, Any]             # TMDB watch/providers payload (e.g., {"results": {"NL": {...}, "US": {...}}})

    # optional control
    next_action: NotRequired[str]
    error: NotRequired[str]

    # optional user context (helpful for final response)
    country: NotRequired[str]             # e.g., "NL"
    subscriptions: NotRequired[List[str]] # e.g., ["Netflix","HBO Max","Amazon Prime Video","Apple TV+"]
    vpn_countries: NotRequired[List[str]] # priority order, e.g., ["US","GB","DE"]

In [17]:
#llm = ChatOllama(model="llama3.1:8b", temperature=0.0)
#llm = ChatOllama(model="gpt-oss:20b", temperature=0.0, keep_alive=False)
from dotenv import load_dotenv
load_dotenv()

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0.0)

TMDB_API_KEY = os.getenv("TMDB_API_KEY") or os.getenv("TMDB_key")

In [19]:
def build_app():
    # ---------- Routers ----------
    def route_after_parse(state: MovieState):
        return END if state.get("error") else "choose_movie_node"
    
    def route_after_choose(state: MovieState):
        return END if state.get("error") else "response_wrap_node"
    
    # ---------- Graph ----------
    graph = StateGraph(MovieState)
    graph.add_node("parse_movie_node", parse_movie_node)
    graph.add_node("choose_movie_node", choose_movie_node)
    graph.add_node("response_wrap_node", response_wrap_node)
    
    graph.set_entry_point("parse_movie_node")
    graph.add_conditional_edges("parse_movie_node", route_after_parse)
    graph.add_conditional_edges("choose_movie_node", route_after_choose)
    graph.add_edge("response_wrap_node", END)
    
    return graph.compile()


In [20]:
#agent = build_app("llama3.1:8b")
agent = build_app()

In [21]:
resp = agent.invoke({"messages":[HumanMessage(content=input("Input movie to watch\n").strip())]})
last_msg = resp["messages"][-1]
display(Markdown(last_msg.content))

Input movie to watch
 Shrek 2


/var/folders/vr/m7mn_w3n333d2h4mg6p4r6y40000gn/T/ipykernel_80284/3388446106.py:9: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.12/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode="after")


NameError: name '_rq' is not defined

In [ ]:
resp = agent.invoke({"messages":[HumanMessage(content=input("Input movie to watch\n").strip())]})
last_msg = resp["messages"][-1]
display(Markdown(last_msg.content))



In [ ]:
resp = agent.invoke({"messages":[HumanMessage(content=input("Input movie to watch\n").strip())]})
last_msg = resp["messages"][-1]
display(Markdown(last_msg.content))